usuarios
- id
- tipo (medico / admin / paciente)
- nombre
- correo
- telefono
- password_hash
- paciente_id            (FK a pacientes, solo si tipo = 'paciente')
- estado_registro (activo/inactivo)

pacientes
- id
- nombre
- documento
- eps
- estado_paciente (activo/inactivo)


solicitudes_traslado
- id
- fecha_hora
- hospital_solicitante
- paciente_id             (FK a pacientes)
- institucion_destino_definitiva
- estado (pendiente/resuelto/fallido)
- estado_proceso (activo/inactivo)

contactos_institucion      (sin soft delete)
- id
- solicitud_id            (FK a solicitudes_traslado)
- institucion_contactada
- fecha_hora_solicitud
- fecha_hora_respuesta
- respuesta (aceptado/rechazado/sin_respuesta)

observaciones
- id
- solicitud_id            (FK a solicitudes_traslado)
- medico_id                (FK a usuarios)
- motivo
- estado_paciente          (texto clínico acerca del estado de salud)
- nivel_urgencia (leve/moderado/crítico)
- estado_proceso (activo/inactivo)

In [1]:
%pip install -q psycopg2-binary pymongo python-dotenv

^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Obtiene la ruta del directorio actual donde está este archivo/notebook
env_path = Path.cwd() / "pass.env"

# Cargar especificando la ruta exacta y sobreescribir variables existentes
loaded = load_dotenv(dotenv_path=env_path, override=True)

# Diagnóstico rápido: load_dotenv devuelve True si encontró y leyó el archivo
print(f"¿Archivo encontrado y cargado?: {loaded}")

PG_CONNECTION_STRING = os.getenv("PG_CONNECTION_STRING")


# Verificación detallada de cuál variable falta
if not PG_CONNECTION_STRING:
    raise ValueError("Falta PG_CONNECTION_STRING en pass.env")


print("Variables cargadas correctamente.")

¿Archivo encontrado y cargado?: True
Variables cargadas correctamente.


##Tablas SQL

In [1]:

DROP TABLE IF EXISTS pacientes;

CREATE TABLE pacientes (
    id SERIAL PRIMARY KEY,
    nombre TEXT NOT NULL,
    documento TEXT NOT NULL UNIQUE,
    eps TEXT NOT NULL,
    activo BOOLEAN NOT NULL DEFAULT TRUE
);

DROP TABLE IF EXISTS usuarios;

CREATE TABLE usuarios (
    id SERIAL PRIMARY KEY,
    tipo TEXT NOT NULL CHECK (tipo IN ('medico', 'admin', 'paciente')),
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    telefono TEXT,
    password_hash TEXT NOT NULL,
    paciente_id INTEGER REFERENCES pacientes(id) ON DELETE SET NULL,
    activo BOOLEAN NOT NULL DEFAULT TRUE
);


DROP TABLE IF EXISTS solicitudes_traslado;

CREATE TABLE solicitudes_traslado (
    id SERIAL PRIMARY KEY,
    fecha_hora TIMESTAMP NOT NULL,
    hospital_solicitante TEXT NOT NULL,
    paciente_id INTEGER NOT NULL REFERENCES pacientes(id) ON DELETE CASCADE,
    institucion_destino_definitiva TEXT NOT NULL,
    estado TEXT NOT NULL CHECK (estado IN ('pendiente', 'resuelto', 'fallido')),
    activo BOOLEAN NOT NULL DEFAULT TRUE
);

DROP TABLE IF EXISTS contactos_institucion;

CREATE TABLE contactos_institucion (
    id SERIAL PRIMARY KEY,
    solicitud_id INTEGER NOT NULL REFERENCES solicitudes_traslado(id) ON DELETE CASCADE,
    institucion_contactada TEXT NOT NULL,
    fecha_hora_solicitud TIMESTAMP NOT NULL,
    fecha_hora_respuesta TIMESTAMP,
    respuesta TEXT CHECK (respuesta IN ('aceptado', 'rechazado', 'sin_respuesta'))
);

DROP TABLE IF EXISTS observaciones;

CREATE TABLE observaciones (
    id SERIAL PRIMARY KEY,
    solicitud_id INTEGER NOT NULL REFERENCES solicitudes_traslado(id) ON DELETE CASCADE,
    medico_id INTEGER NOT NULL REFERENCES usuarios(id) ON DELETE CASCADE,
    motivo TEXT NOT NULL,
    estado_paciente TEXT NOT NULL,
    nivel_urgencia TEXT NOT NULL CHECK (nivel_urgencia IN ('leve', 'moderado', 'critico')),
    activo BOOLEAN NOT NULL DEFAULT TRUE
);


SyntaxError: invalid syntax (307577163.py, line 1)